# Paired daily diffusion v2

Run from this repository in VS Code or Colab. The implementation lives in `src/ausgrid_synth/diffusion.py`; this notebook runs and reviews its checkpoints. The training and sampling cells require a GPU for practical runtime. `prepared.npz` must first be created by the original preparation stage. The model emits independent paired GC+CL and GG daily profiles, not persistent households.

The first diffusion experiment (`diffusion`, `diffusion_post`) had extreme load peaks and is retained only as a failed historical result. This notebook uses `diffusion_v2` and the solar-night-masked `diffusion_v2_post`. Keep the fixed household split and train-only transform. The held-out test has already informed later work, so comparisons are exploratory.

## 1. Project and data

In [ ]:
from pathlib import Path
import os, sys, subprocess, json
import numpy as np
import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
candidates = (cwd, cwd.parent, cwd/'ausgrid-synthetic',
              Path('/content/drive/MyDrive/ausgrid-synthetic/ausgrid-synthetic'),
              Path('/content/drive/MyDrive/ausgrid-synthetic'))
PROJECT_ROOT = next((p for p in candidates if (p/'src/ausgrid_synth/cli.py').is_file()
                     and (p/'data/prepared/prepared.npz').is_file()), None)
if PROJECT_ROOT is None and 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    PROJECT_ROOT = next((p for p in candidates if (p/'src/ausgrid_synth/cli.py').is_file()
                         and (p/'data/prepared/prepared.npz').is_file()), None)
assert PROJECT_ROOT is not None, 'Locate the project with src/ and prepared.npz, or run the preparation notebook first.'
os.environ['PYTHONPATH'] = str(PROJECT_ROOT/'src')
if os.environ['PYTHONPATH'] not in sys.path: sys.path.insert(0,os.environ['PYTHONPATH'])
from ausgrid_synth.data import load_prepared
from ausgrid_synth.cli import context_indices
DATA = load_prepared(PROJECT_ROOT/'data/prepared/prepared.npz')
SEEDS = (1,2,3)
REPORTS = PROJECT_ROOT/'outputs/reports'
SAMPLES = PROJECT_ROOT/'outputs/samples'
print('Project:', PROJECT_ROOT, '| split households:',
      {name:len(np.unique(DATA['customer'][DATA['split']==code]))
       for code,name in enumerate(('train','validation','test'))})

def run(stage, arm, seed=1, **options):
    cmd=[sys.executable,'-m','ausgrid_synth.cli',stage,'--root',str(PROJECT_ROOT),
         '--arm',arm,'--seed',str(seed)]
    for key,value in options.items(): cmd.extend(['--'+key.replace('_','-'),str(value)])
    subprocess.run(cmd,check=True,env=os.environ.copy())


## 2. Train seed 1 and check validation peaks

The optional two-epoch pilot resumes into the 25-epoch run. Checkpoints are tied to the prepared-data hash and fixed configuration. Reruns resume a matching checkpoint. The validation peak gate runs before full sampling or reading test outcomes.

In [ ]:
import torch
assert torch.cuda.is_available(), 'Select a GPU kernel for diffusion training and sampling.'
run('train','diffusion_v2',seed=1,epochs=2)
run('train','diffusion_v2',seed=1,epochs=25)
run('validate','diffusion_v2',seed=1)


## 3. Finish seeds and generate matched contexts

In [ ]:
for seed in (2,3): run('train','diffusion_v2',seed=seed,epochs=25)
for seed in SEEDS:
    for context in ('train','test','privacy'):
        run('sample','diffusion_v2_post',seed=seed,context=context)


## 4. Evaluate fixed reports

Each report uses the unchanged VAE evaluator. Lower W1 and MAE are favorable. The attack AUC is a bounded empirical probe and cannot establish a privacy guarantee. Three seeds share one household split.

In [ ]:
for seed in SEEDS: run('evaluate','diffusion_v2_post',seed=seed)
def report_table(arms):
    rows=[]
    for arm in arms:
        for seed in SEEDS:
            path=REPORTS/f'{arm}_seed{seed}.json'
            if not path.exists(): continue
            r=json.loads(path.read_text()); f=r['fidelity_and_physics']; u=r['utility']; p=r['disclosure_probe']
            rows.append({'arm':arm,'seed':seed,
                         'load_daily_W1':f['total_consumption_daily_kwh_wasserstein'],
                         'load_peak_W1':f['total_consumption_peak_interval_kwh_wasserstein'],
                         'solar_daily_W1':f['gross_generation_daily_kwh_wasserstein'],
                         'night_GG_kWh':f['night_solar_kwh'],
                         'synthetic_to_real_MAE':u['synthetic_train_test_real_mae'],
                         'real_to_real_MAE':u['real_train_test_real_mae'],
                         'attack_AUC':p['household_membership_auc']})
    table=pd.DataFrame(rows)
    display(table.round(4))
    if len(table): display(table.groupby('arm').mean(numeric_only=True).drop(columns='seed').round(4))
    return table
results=report_table(('vae_post','diffusion_v2_post'))


## 5. Profile and peak audit

Inspect daily shape and peak tails as well as aggregate scores. The night mask sets modeled nighttime GG to zero; the physical rule uses an approximate Sydney solar elevation and is not roof-level PV simulation.

In [ ]:
import matplotlib.pyplot as plt
with np.load(SAMPLES/'diffusion_v2_post_seed1_test.npz',allow_pickle=False) as z:
    synthetic,indices=z['x'],z['idx']
assert np.array_equal(indices,context_indices(DATA,'test'))
real=DATA['x'][indices]
fig,axes=plt.subplots(1,2,figsize=(12,3.5))
for channel,label in enumerate(('GC+CL','GG')):
    print(label,'real/generated peak q95, q99:',
          np.quantile(real[:,channel].max(axis=1),[.95,.99]).round(3),
          np.quantile(synthetic[:,channel].max(axis=1),[.95,.99]).round(3))
    hour=np.arange(48)/2+.25
    axes[channel].plot(hour,np.median(real[:,channel],axis=0),label='held-out real')
    axes[channel].plot(hour,np.median(synthetic[:,channel],axis=0),label='diffusion v2 + mask')
    axes[channel].set(xlabel='Local hour',ylabel='kWh per half-hour',title=label)
    axes[channel].legend()
plt.tight_layout();plt.show()


## Decision boundary

The recorded three-seed comparison improves peak W1 and downstream utility over VAE, while VAE has a lower daily load W1. Use `Ausgrid_Diffusion_Load_Calibration.ipynb` for an optional exploratory mapping of daily load totals. Do not distribute prepared data or sample archives containing `idx` as anonymous public outputs.